# 🫁 Pneumonia Detection with PneumoniaMNIST

This notebook demonstrates the pneumonia detection pipeline using a **pretrained ResNet18** model.

- Dataset: **PneumoniaMNIST** (MedMNIST v2)
- Task: Binary classification (Normal vs Pneumonia)
- Image size: 28×28 grayscale

## 1. Setup & Installation

In [ ]:
!pip install -q torch torchvision medmnist numpy pandas matplotlib seaborn scikit-learn tqdm pyyaml

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from tqdm.notebook import tqdm
import medmnist
from medmnist import PneumoniaMNIST
from torchvision import transforms as T, models

# Set seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Download & Prepare Dataset

In [ ]:
# Transforms
train_transform = T.Compose([
    T.Resize((28, 28)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=10),
    T.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize([0.5], [0.5]),
])

eval_transform = T.Compose([
    T.Resize((28, 28)),
    T.ToTensor(),
    T.Normalize([0.5], [0.5]),
])

# Load datasets
train_dataset = PneumoniaMNIST(split='train', download=True, transform=train_transform)
val_dataset = PneumoniaMNIST(split='val', download=True, transform=eval_transform)
test_dataset = PneumoniaMNIST(split='test', download=True, transform=eval_transform)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

# Create dataloaders
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

### Visualize Sample Images

In [ ]:
class_names = ['Normal', 'Pneumonia']

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for i, ax in enumerate(axes.flatten()):
    img, label = test_dataset[i]
    img = img.squeeze().numpy() * 0.5 + 0.5
    ax.imshow(img, cmap='gray')
    ax.set_title(class_names[int(label.squeeze())], fontsize=10)
    ax.axis('off')
plt.suptitle('Sample Images from PneumoniaMNIST', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Model: ResNet18 (Pretrained)

In [ ]:
class ResNetClassifier(nn.Module):
    def __init__(self, in_channels=1, pretrained=True):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        self.model = models.resnet18(weights=weights)

        # Adapt for grayscale
        orig_conv = self.model.conv1
        self.model.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        if pretrained:
            self.model.conv1.weight.data = orig_conv.weight.data.mean(dim=1, keepdim=True)

        # Binary output
        in_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_features, 1))

    def forward(self, x):
        return self.model(x)

model = ResNetClassifier(pretrained=True).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')

## 4. Training

In [ ]:
# Training setup
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

EPOCHS = 15
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_auc': []}
best_auc = 0.0

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False):
        images = images.to(device)
        labels = labels.float().to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    train_loss /= total
    train_acc = correct / total

    # Validate
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    all_labels, all_probs = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.float().to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_labels.extend(labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
    val_loss /= total
    val_acc = correct / total
    val_auc = roc_auc_score(all_labels, all_probs)

    scheduler.step(val_auc)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    if val_auc > best_auc:
        best_auc = val_auc
        best_state = model.state_dict().copy()
        marker = ' ⭐ Best'
    else:
        marker = ''

    print(f'Epoch {epoch:2d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}{marker}')

# Load best model
model.load_state_dict(best_state)
print(f'\nBest Validation AUC: {best_auc:.4f}')

## 5. Training Curves

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs, history['train_loss'], 'b-', label='Train')
axes[0].plot(epochs, history['val_loss'], 'r-', label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history['train_acc'], 'b-', label='Train')
axes[1].plot(epochs, history['val_acc'], 'r-', label='Val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, history['val_auc'], 'g-', label='Val AUC')
axes[2].set_title('Validation AUC'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Training Curves — ResNet18 Pretrained', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Evaluation on Test Set

In [ ]:
model.eval()
all_labels, all_probs, all_images = [], [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Testing'):
        images = images.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs).cpu().numpy().flatten()
        all_labels.extend(labels.numpy().flatten())
        all_probs.extend(probs)
        all_images.append(images.cpu())

true_labels = np.array(all_labels)
probabilities = np.array(all_probs)
pred_labels = (probabilities >= 0.5).astype(int)
all_images = torch.cat(all_images, dim=0)

# Metrics
print('\n' + '='*50)
print('TEST SET RESULTS')
print('='*50)
print(f"Accuracy:  {accuracy_score(true_labels, pred_labels):.4f}")
print(f"Precision: {precision_score(true_labels, pred_labels):.4f}")
print(f"Recall:    {recall_score(true_labels, pred_labels):.4f}")
print(f"F1 Score:  {f1_score(true_labels, pred_labels):.4f}")
print(f"ROC AUC:   {roc_auc_score(true_labels, probabilities):.4f}")
print(f'\n{classification_report(true_labels, pred_labels, target_names=class_names)}')

## 7. Confusion Matrix

In [ ]:
cm = confusion_matrix(true_labels, pred_labels)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — ResNet18 Pretrained')
plt.tight_layout()
plt.show()

## 8. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(true_labels, probabilities)
auc_score = roc_auc_score(true_labels, probabilities)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='#2563eb', lw=2, label=f'ROC (AUC = {auc_score:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — ResNet18 Pretrained')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Sample Predictions

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
indices = np.random.choice(len(true_labels), 12, replace=False)

for i, (ax, idx) in enumerate(zip(axes.flatten(), indices)):
    img = all_images[idx].squeeze().numpy() * 0.5 + 0.5
    ax.imshow(img, cmap='gray')
    true_cls = class_names[int(true_labels[idx])]
    pred_cls = class_names[int(pred_labels[idx])]
    prob = probabilities[idx]
    color = 'green' if true_cls == pred_cls else 'red'
    ax.set_title(f'T:{true_cls}\nP:{pred_cls} ({prob:.2f})', fontsize=9, color=color)
    ax.axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', fontsize=13)
plt.tight_layout()
plt.show()

---

✅ **Demo Complete!** To run all 6 experiments, use: `python -m task1_classification.experiment_runner`